只保留核心逻辑：dual-clip + stop-gradient(sg)

In [1]:
import torch
import torch.nn.functional as F

def sg(x):
    """
    stop-gradient：前向为x，反向为0
    
    前向用裁剪后的权重，反向不传播 w_clip 的梯度，只保留 ∇logπ
    """
    return x.detach() - x + x

# CISPO 梯度计算
def cispo_gradient(logits_new,logits_old,action,adv,tau_high=5.0):
    """
    logits_new: [B, L, V]当前策略输出
    logtis_old: [B, L, V]旧策略输出
    action    : [B, L]采样的token-id
    tau_high  : [B, L]硬裁剪上界
    adv       : 优势
    return    : 每张token的梯度张量，后续可接optimizer
    """
    B,L,V=logits_new.shape
    
    # 计算重要性权重 w = π_new / π_old
    logp_new = -F.cross_entropy(
        logits_new.view(-1,V),action.view(-1),reduction='none'
    ).view(B,L)
    logp_old = -F.cross_entropy(
        logtis_old.view(-1,V),action.view(-1),reduction='none'
    ).view(B,L)
    logp_old = logp_old.detach() # 保证旧策略不计算梯度
    log_w = logp_new - logp_old
    w = log_w.exp()  # [B,L]
    
    # 2 硬裁剪 & 标记dual-clip
    dual_clipped = w > tau_high
    w_clip = torch.clamp(w,max=tau_high)
    
    # 3 软裁剪梯度
    # 对 dual_clipped token 用sg(w_clip) * ∇logπ (对原本使用硬裁剪的token，使用软裁剪)
    # 其余token使用普通的 w * ∇logπ
    
    weight = torch.where(dual_clipped,sg(w_clip),w)
    
    # 4 每个token的梯度 = -weigth * adv * ∇logπ
    # 负号- 因为pytorch默认是minimize
    loss_per_tokne = -weight * adv
    loss = loss_per_token.mean()
    
    # 返回的 loss 直接喂给 optimizer.step() 即可，与 HF 的 transformers 无缝兼容
    return loss,{
        'w':w,
        'w_clip':w_clip,
        'dual_clipped_mask':dual_clipped, # 记录哪些 token 被软裁剪，可用于日志或可视化
    }


    

In [ ]:
# ---------使用示例 -------------
"""
Batch = 2　         # 两条句子
Length = 10         # 每条 10 个 token
Vocab = 50 257        # GPT-2 词表大小
"""
B,L,V = 2,10,50257
logits_new = torch.randn(B,L,V,requires_grad = True)
# 在真实代码里，这里应是上一轮模型的输出，通常先存下来再 detach
logits_old = torch.randn(B,L,V).detach()
action = torch.randint(0,V,(B,L))
adv = torch.randn(B,L)

loss,info = cispo_gradient(logits_new,logits_old,action,adv,tau_high=5.0)
loss.backward()
print('dual_clipped ratio:',info['dual_clipped_mask'].float().mean().item())

## 扩展到完整 PPO/GRPO 脚本:
- 把 cispo_gradient 替换原 PPO 的 ratio * adv 部分即可。
- 其余环节（value loss、KL penalty、entropy bonus、global grad-norm）保持不变。

这样，低概率但关键的“分叉” token 既能被充分学习，又不会把梯度炸掉。